In [1]:
# ── 1. Install ──────────────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations rasterio timm einops
!pip install -q transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [2]:
# ── 2. Imports & Reproducibility ────────────────────────────────────────────
import os, gc, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
import albumentations as A
import segmentation_models_pytorch as smp
from tqdm import tqdm
import cv2
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"CUDA " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

PyTorch  : 2.10.0+cu128
Device   : CUDA Tesla T4


In [3]:
# ── 3. Config ────────────────────────────────────────────────────────────────
import torch

class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────
    BASE_PATH = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
    TRAIN_TXT = f'{BASE_PATH}/split/train.txt'
    VAL_TXT   = f'{BASE_PATH}/split/val.txt'
    TEST_TXT  = f'{BASE_PATH}/split/test.txt'
    IMG_DIR   = f'{BASE_PATH}/image'
    LBL_DIR   = f'{BASE_PATH}/label'
    TEST_DIR  = f'{BASE_PATH}/prediction/image'
    SAVE_DIR  = '/kaggle/working'

    # ── Model ─────────────────────────────────────────────────────────────
    IN_CHANNELS = 6       # HH, HV, G, R, NIR, SWIR  (maps to Prithvi 6-band HLS)
    NUM_CLASSES = 3
    IMG_SIZE    = 512

    # ── Training ──────────────────────────────────────────────────────────
    BATCH_SIZE       = 2    # smaller — Prithvi is large
    EPOCHS           = 25
    LR               = 5e-5  # low LR for fine-tuning pretrained backbone
    WARMUP_EPOCHS    = 3
    WEIGHT_DECAY     = 1e-4
    GRAD_CLIP        = 1.0
    ACCUMULATE_STEPS = 4    # effective batch = 8

    # ── Class Weights ─────────────────────────────────────────────────────
    CLASS_WEIGHTS = [1.0, 10.0, 3.0]

    # ── Inference ─────────────────────────────────────────────────────────
    TTA_IDS = [0, 1, 2, 3, 4, 5, 6, 7]   # all 8 transforms
    DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── Prithvi band normalization (HLS pretraining stats) ────────────────
    # Order: Blue, Green, Red, NIR, SWIR1, SWIR2
    # We map: HH→Blue, HV→Green, G→Red, R→NIR, NIR→SWIR1, SWIR→SWIR2
    PRITHVI_MEAN = [0.033, 0.085, 0.072, 0.277, 0.233, 0.143]
    PRITHVI_STD  = [0.021, 0.026, 0.032, 0.062, 0.057, 0.038]

print('Config loaded ✅')

Config loaded ✅


In [4]:
# ── 4. List test files ───────────────────────────────────────────────────────
import os
for f in sorted(os.listdir(CFG.TEST_DIR)):
    print(f)

20240529_EO4_RES2_fl_pid_080_image.tif
20240529_EO4_RES2_fl_pid_081_image.tif
20240529_EO4_RES2_fl_pid_082_image.tif
20240529_EO4_RES2_fl_pid_083_image.tif
20240529_EO4_RES2_fl_pid_084_image.tif
20240529_EO4_RES2_fl_pid_085_image.tif
20240529_EO4_RES2_fl_pid_086_image.tif
20240529_EO4_RES2_fl_pid_087_image.tif
20240529_EO4_RES2_fl_pid_088_image.tif
20240529_EO4_RES2_fl_pid_089_image.tif
20240529_EO4_RES2_fl_pid_090_image.tif
20240529_EO4_RES2_fl_pid_091_image.tif
20240529_EO4_RES2_fl_pid_092_image.tif
20240529_EO4_RES2_fl_pid_093_image.tif
20240529_EO4_RES2_fl_pid_094_image.tif
20240529_EO4_RES2_fl_pid_095_image.tif
20240529_EO4_RES2_fl_pid_096_image.tif
20240529_EO4_RES2_fl_pid_097_image.tif
20240529_EO4_RES2_fl_pid_098_image.tif


In [5]:
# ── 5. Data Loading ──────────────────────────────────────────────────────────
def load_split(file_path, img_dir=CFG.IMG_DIR, lbl_dir=CFG.LBL_DIR, with_labels=True):
    with open(file_path, 'r') as f:
        ids = [l.strip() for l in f.readlines() if l.strip()]
    images = [os.path.join(img_dir, i + '_image.tif') for i in ids]
    if with_labels:
        labels = [os.path.join(lbl_dir, i + '_label.tif') for i in ids]
        return images, labels, ids
    return images, None, ids


train_imgs, train_lbls, train_ids = load_split(CFG.TRAIN_TXT)
val_imgs,   val_lbls,   val_ids   = load_split(CFG.VAL_TXT)
lbl_test_imgs, lbl_test_lbls, lbl_test_ids = load_split(CFG.TEST_TXT)

test_ids = [
    '20240529_EO4_RES2_fl_pid_080', '20240529_EO4_RES2_fl_pid_081',
    '20240529_EO4_RES2_fl_pid_082', '20240529_EO4_RES2_fl_pid_083',
    '20240529_EO4_RES2_fl_pid_084', '20240529_EO4_RES2_fl_pid_085',
    '20240529_EO4_RES2_fl_pid_086', '20240529_EO4_RES2_fl_pid_087',
    '20240529_EO4_RES2_fl_pid_088', '20240529_EO4_RES2_fl_pid_089',
    '20240529_EO4_RES2_fl_pid_090', '20240529_EO4_RES2_fl_pid_091',
    '20240529_EO4_RES2_fl_pid_092', '20240529_EO4_RES2_fl_pid_093',
    '20240529_EO4_RES2_fl_pid_094', '20240529_EO4_RES2_fl_pid_095',
    '20240529_EO4_RES2_fl_pid_096', '20240529_EO4_RES2_fl_pid_097',
    '20240529_EO4_RES2_fl_pid_098',
]
test_imgs = [os.path.join(CFG.TEST_DIR, f'{tid}_image.tif') for tid in test_ids]

print(f'Train: {len(train_imgs)} | Val: {len(val_imgs)} | LabeledTest: {len(lbl_test_imgs)} | PredTest: {len(test_imgs)}')
missing = [p for p in test_imgs if not os.path.exists(p)]
print(f'Missing: {missing}' if missing else '✅ All 19 test files found')

Train: 59 | Val: 10 | LabeledTest: 10 | PredTest: 19
✅ All 19 test files found


In [6]:
def preprocess(image):
    # Standard input: [SAR_HH, SAR_HV, Green, Red, NIR, SWIR]
    HH, HV, G, R, NIR, SWIR = image

    # 1. SAR Log Transform
    HH_log = np.log1p(np.clip(HH, 0, None))
    HV_log = np.log1p(np.clip(HV, 0, None))
    
    # 2. Spectral Alignment for Prithvi-100M
    # Pretraining order: Blue, Green, Red, Narrow NIR, SWIR1, SWIR2
    # We map HH to Blue and HV to Green to provide 'texture' where color is missing
    out = np.stack([
        HH_log,  # Proxy for Blue
        HV_log,  # Proxy for Green
        G,       # Red slot (Green band)
        R,       # NIR slot (Red band)
        NIR,     # SWIR1 slot (NIR band)
        SWIR,    # SWIR2 slot (SWIR band)
    ], axis=0)

    # 3. Percentile Clipping (Robust Scaling)
    for c in range(6):
        lo, hi = np.percentile(out[c], 2), np.percentile(out[c], 98)
        out[c] = np.clip((out[c] - lo) / (hi - lo + 1e-6), 0, 1)

    # 4. Apply Prithvi HLS Stats
    mean = np.array(CFG.PRITHVI_MEAN, dtype=np.float32).reshape(6, 1, 1)
    std  = np.array(CFG.PRITHVI_STD,  dtype=np.float32).reshape(6, 1, 1)
    out  = (out - mean) / (std + 1e-8)

    return out.astype(np.float32)

In [7]:
# ── 7. Dataset ───────────────────────────────────────────────────────────────
class FloodDataset(Dataset):
    def __init__(self, imgs, lbls=None, transform=None):
        self.imgs      = imgs
        self.lbls      = lbls
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def _clip_outliers(self, image, pct=2):
        for c in range(image.shape[0]):
            lo = np.percentile(image[c], pct)
            hi = np.percentile(image[c], 100 - pct)
            image[c] = np.clip(image[c], lo, hi)
        return image

    def __getitem__(self, idx):
        with rasterio.open(self.imgs[idx]) as f:
            image = f.read().astype(np.float32)
        image = image[:, :CFG.IMG_SIZE, :CFG.IMG_SIZE]
        image = self._clip_outliers(image)
        image = preprocess(image)              # (6, H, W)
        image = np.transpose(image, (1, 2, 0)) # (H, W, 6) for albumentations

        if self.lbls is not None:
            with rasterio.open(self.lbls[idx]) as f:
                label = f.read(1).astype(np.int64)
            label = label[:CFG.IMG_SIZE, :CFG.IMG_SIZE]
            label = np.clip(label, 0, CFG.NUM_CLASSES - 1)
        else:
            label = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.int64)

        if self.transform:
            aug   = self.transform(image=image, mask=label)
            image = aug['image']
            label = aug['mask']

        image = np.transpose(image, (2, 0, 1))  # back to (6, H, W)
        return torch.tensor(image, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

print('FloodDataset defined ✅')

FloodDataset defined ✅


In [8]:
# ── 8. Augmentations ─────────────────────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08),
             rotate=(-45, 45), shear=(-5, 5), p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    A.GridDistortion(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.Blur(blur_limit=3, p=0.15),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    min_holes=1, fill_value=0, p=0.2),
])
val_transform = A.Compose([])
print('Augmentations defined ✅')

Augmentations defined ✅


In [9]:
# ── 9. Prithvi-100M Segmentation Model (FULL MANUAL LOAD) ───────────────────────────────────────
class ConvBnGelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, pad=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
    def forward(self, x): return self.block(x)

class PatchEmbed(nn.Module):
    """Custom patch embedding to replace the transformer's embedder"""
    def __init__(self, img_size=512, patch_size=16, in_chans=6, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, E, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, n_patches, E)
        return x

class PrithviSegmentation(nn.Module):
    """
    Complete manual implementation to avoid HuggingFace config issues.
    Uses the pretrained weights but builds the model structure manually.
    """
    HIDDEN_DIM = 768
    PATCH_SIZE = 16
    NUM_LAYERS = 12
    NUM_HEADS = 12
    
    def __init__(self, num_classes=3, img_size=512):
        super().__init__()
        from huggingface_hub import hf_hub_download
        import torch
        import json
        
        self.img_size = img_size
        self.n_patch = img_size // self.PATCH_SIZE
        
        print(f'Loading Prithvi-100M weights manually...')
        
        # Download the weights
        REPO_ID = 'ibm-nasa-geospatial/Prithvi-EO-1.0-100M'
        weights_path = hf_hub_download(repo_id=REPO_ID, filename="Prithvi_100M.pt")
        
        # Load state dict
        state_dict = torch.load(weights_path, map_location='cpu')
        
        # Build the model structure
        self.patch_embed = PatchEmbed(img_size, self.PATCH_SIZE, 6, self.HIDDEN_DIM)
        
        # Position embeddings
        n_patches = self.n_patch ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, self.HIDDEN_DIM))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, self.HIDDEN_DIM))
        
        # Transformer encoder layers
        from torch.nn import TransformerEncoder, TransformerEncoderLayer
        encoder_layer = TransformerEncoderLayer(
            d_model=self.HIDDEN_DIM,
            nhead=self.NUM_HEADS,
            dim_feedforward=self.HIDDEN_DIM * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.encoder = TransformerEncoder(encoder_layer, self.NUM_LAYERS)
        
        self.norm = nn.LayerNorm(self.HIDDEN_DIM)
        self.drop = nn.Dropout(0.1)
        
        # Decoder (simple but effective)
        self.decoder = nn.Sequential(
            nn.Conv2d(self.HIDDEN_DIM, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(64, num_classes, kernel_size=1)
        )
        
        # Load pretrained weights (map from the original model to our structure)
        self._load_pretrained_weights(state_dict)
        
        print('Prithvi-100M loaded successfully ✅')
    
    def _load_pretrained_weights(self, state_dict):
        """Map pretrained weights to our model structure"""
        # Map patch embedding weights
        if 'vit.embeddings.patch_embeddings.projection.weight' in state_dict:
            self.patch_embed.proj.weight.data = state_dict['vit.embeddings.patch_embeddings.projection.weight']
            self.patch_embed.proj.bias.data = state_dict['vit.embeddings.patch_embeddings.projection.bias']
        
        # Map position embeddings
        if 'vit.embeddings.position_embeddings.weight' in state_dict:
            pos_weight = state_dict['vit.embeddings.position_embeddings.weight']
            self.pos_embed.data = pos_weight.unsqueeze(0)
        
        # Map CLS token
        if 'vit.embeddings.cls_token' in state_dict:
            self.cls_token.data = state_dict['vit.embeddings.cls_token']
        
        # Map transformer layers
        for i in range(self.NUM_LAYERS):
            prefix = f'vit.encoder.layer.{i}'
            target_prefix = f'encoder.layers.{i}'
            
            # Attention weights
            if f'{prefix}.attention.self.query.weight' in state_dict:
                self.encoder.layers[i].self_attn.in_proj_weight.data = torch.cat([
                    state_dict[f'{prefix}.attention.self.query.weight'],
                    state_dict[f'{prefix}.attention.self.key.weight'],
                    state_dict[f'{prefix}.attention.self.value.weight']
                ], dim=0)
                self.encoder.layers[i].self_attn.in_proj_bias.data = torch.cat([
                    state_dict[f'{prefix}.attention.self.query.bias'],
                    state_dict[f'{prefix}.attention.self.key.bias'],
                    state_dict[f'{prefix}.attention.self.value.bias']
                ], dim=0)
            
            # FFN weights
            if f'{prefix}.intermediate.dense.weight' in state_dict:
                self.encoder.layers[i].linear1.weight.data = state_dict[f'{prefix}.intermediate.dense.weight']
                self.encoder.layers[i].linear1.bias.data = state_dict[f'{prefix}.intermediate.dense.bias']
                self.encoder.layers[i].linear2.weight.data = state_dict[f'{prefix}.output.dense.weight']
                self.encoder.layers[i].linear2.bias.data = state_dict[f'{prefix}.output.dense.bias']
            
            # Layer norms
            if f'{prefix}.attention.output.LayerNorm.weight' in state_dict:
                self.encoder.layers[i].norm1.weight.data = state_dict[f'{prefix}.attention.output.LayerNorm.weight']
                self.encoder.layers[i].norm1.bias.data = state_dict[f'{prefix}.attention.output.LayerNorm.bias']
                self.encoder.layers[i].norm2.weight.data = state_dict[f'{prefix}.output.LayerNorm.weight']
                self.encoder.layers[i].norm2.bias.data = state_dict[f'{prefix}.output.LayerNorm.bias']
        
        # Final layer norm
        if 'vit.encoder.layernorm.weight' in state_dict:
            self.norm.weight.data = state_dict['vit.encoder.layernorm.weight']
            self.norm.bias.data = state_dict['vit.encoder.layernorm.bias']
        
        print('Pretrained weights loaded ✅')
    
    def forward(self, x):
        B = x.shape[0]
        n = self.n_patch
        
        # Patch embedding
        x = self.patch_embed(x)  # (B, n_patches, HIDDEN_DIM)
        
        # Add CLS token and position embeddings
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        
        # Transformer encoder
        x = self.encoder(x)
        
        # Remove CLS token and reshape
        tokens = x[:, 1:, :]  # Remove CLS token
        tokens = tokens[:, :n*n, :]  # Ensure exact size
        
        # Reshape to spatial feature map
        feat = tokens.permute(0, 2, 1).reshape(B, self.HIDDEN_DIM, n, n)
        feat = self.drop(feat)
        
        # Decode to segmentation map
        return self.decoder(feat)

print('PrithviSegmentation (Complete Manual Implementation) defined ✅')

PrithviSegmentation (Complete Manual Implementation) defined ✅


In [10]:
# ── 10. Backup SMP Models (for ensemble) ─────────────────────────────────────

def build_smp_model_1():
    """UNet++ EfficientNet-B5 — 6 input channels."""
    return smp.UnetPlusPlus(
        encoder_name='efficientnet-b5',
        encoder_weights='imagenet',
        in_channels=6,
        classes=CFG.NUM_CLASSES,
        activation=None,
        decoder_attention_type='scse',
    )

def build_smp_model_2():
    """DeepLabV3+ ResNet50 — 6 input channels."""
    return smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=6,
        classes=CFG.NUM_CLASSES,
        activation=None,
        encoder_output_stride=16,
    )

print('SMP model builders defined ✅  (UNet++ B5 + DeepLabV3+ R50, 6-ch input)')

SMP model builders defined ✅  (UNet++ B5 + DeepLabV3+ R50, 6-ch input)


In [11]:
# ── 11. Lovász + Focal Loss ──────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F


def lovasz_grad(gt_sorted):
    p    = len(gt_sorted)
    gts  = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jac   = 1.0 - inter / union
    if p > 1:
        jac[1:p] = jac[1:p] - jac[0:-1]
    return jac


def lovasz_softmax_flat(probs, labels, classes='present'):
    if probs.numel() == 0:
        return probs * 0.
    C, losses = probs.size(1), []
    for c in range(C):
        fg = (labels == c).float()
        if classes == 'present' and fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm.data]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))
    return torch.stack(losses).mean()


def lovasz_softmax(preds, targets):
    probs = torch.softmax(preds, dim=1)
    B, C, H, W = probs.shape
    return lovasz_softmax_flat(
        probs.permute(0, 2, 3, 1).contiguous().view(-1, C),
        targets.view(-1)
    )


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, class_weights=None):
        super().__init__()
        self.gamma, self.class_weights = gamma, class_weights

    def forward(self, preds, targets):
        ce    = F.cross_entropy(preds, targets, weight=self.class_weights, reduction='none')
        focal = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        return focal.mean()


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Heavily weight Class 1 (Flood) to force the model to find it
        self.ce = nn.CrossEntropyLoss(
            weight=torch.tensor([1.0, 15.0, 5.0], device=CFG.DEVICE)
        )

    def forward(self, preds, targets):
        # CrossEntropy handles the 0, 1, 2 labels for Phase 2
        return self.ce(preds, targets)

criterion = CombinedLoss()
print('Loss: 0.6×Lovász + 0.4×Focal(γ=2) ✅')

Loss: 0.6×Lovász + 0.4×Focal(γ=2) ✅


In [12]:
# ── 12. Metrics ──────────────────────────────────────────────────────────────
class SegmentationMetrics:
    def __init__(self, num_classes=CFG.NUM_CLASSES):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.conf_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, preds, targets):
        p = torch.argmax(preds, dim=1).cpu().numpy().flatten()
        t = targets.cpu().numpy().flatten()
        valid = (t >= 0) & (t < self.num_classes)
        idx   = self.num_classes * t[valid] + p[valid]
        self.conf_matrix += np.bincount(idx, minlength=self.num_classes**2).reshape(self.num_classes, self.num_classes)

    def compute(self):
        cm  = self.conf_matrix
        tp  = np.diag(cm)
        iou = np.where((tp + (cm.sum(1)-tp) + (cm.sum(0)-tp)) > 0,
                       tp / (tp + (cm.sum(1)-tp) + (cm.sum(0)-tp)), 0.0)
        return {
            'iou_noflood'  : iou[0],
            'iou_flood'    : iou[1],
            'iou_waterbody': iou[2],
            'mIoU'         : iou.mean(),
            'accuracy'     : tp.sum() / cm.sum() if cm.sum() > 0 else 0.0,
        }

print('Metrics defined ✅')

Metrics defined ✅


In [13]:
# ── 13. Threshold Search ──────────────────────────────────────────────────────
def find_best_threshold(models_list, weights_list, val_loader, device):
    for m in models_list: m.eval()

    all_flood_probs, all_gt_flood = [], []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            logits_sum = None
            for model, w in zip(models_list, weights_list):
                logits = model(images)
                logits_sum = w * logits if logits_sum is None else logits_sum + w * logits
            probs = torch.softmax(logits_sum, dim=1)[:, 1].cpu().numpy()
            all_flood_probs.append(probs.reshape(-1))
            all_gt_flood.append((masks.numpy() == 1).reshape(-1))

    probs_all = np.concatenate(all_flood_probs)
    gt_all    = np.concatenate(all_gt_flood)

    best_t, best_iou = 0.5, 0.0
    for t in np.linspace(0.20, 0.80, 25):
        preds = (probs_all > t).astype(np.uint8)
        tp    = int((preds & gt_all).sum())
        fp    = int((preds & ~gt_all).sum())
        fn    = int((~preds.astype(bool) & gt_all).sum())
        u     = tp + fp + fn
        iou   = tp / u if u > 0 else 0.0
        if iou > best_iou:
            best_iou, best_t = iou, float(t)

    print(f'Best Threshold: {best_t:.3f} | Global Flood IoU: {best_iou:.4f}')
    return best_t

print('find_best_threshold() defined ✅')

find_best_threshold() defined ✅


In [14]:
# ── 14. Scheduler ────────────────────────────────────────────────────────────
def get_scheduler(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / float(max(1, num_warmup_steps))
        progress = float(step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print('Scheduler defined ✅')

Scheduler defined ✅


In [15]:
# ── 15. Training Loop ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(tqdm(loader, desc='Train', leave=False)):
        imgs, masks = imgs.to(CFG.DEVICE), masks.to(CFG.DEVICE)
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs), masks) / CFG.ACCUMULATE_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % CFG.ACCUMULATE_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        total_loss += loss.item() * CFG.ACCUMULATE_STEPS

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss, metrics = 0.0, SegmentationMetrics()
    for imgs, masks in tqdm(loader, desc='Val', leave=False):
        imgs, masks = imgs.to(CFG.DEVICE), masks.to(CFG.DEVICE)
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = criterion(out, masks)
        total_loss += loss.item()
        metrics.update(out, masks)
    r = metrics.compute()
    r['val_loss'] = total_loss / len(loader)
    return r


def train_model(model, model_name, save_name, tr_loader, va_loader,
                lr=CFG.LR, frozen_epochs=3):
    """
    frozen_epochs: freeze Prithvi backbone for first N epochs,
                   then unfreeze for fine-tuning.
                   Set to 0 for SMP models (no backbone to freeze).
    """
    print(f"\n{'='*60}")
    print(f'  Training: {model_name}')
    print(f"{'='*60}")

    # ── Freeze backbone for warm-up if Prithvi ───────────────────────────
    is_prithvi = hasattr(model, 'backbone')
    if is_prithvi and frozen_epochs > 0:
        for p in model.backbone.parameters():
            p.requires_grad = False
        print(f'  Backbone frozen for first {frozen_epochs} epochs')

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CFG.WEIGHT_DECAY
    )
    total_steps  = (len(tr_loader) // CFG.ACCUMULATE_STEPS) * CFG.EPOCHS
    warmup_steps = (len(tr_loader) // CFG.ACCUMULATE_STEPS) * CFG.WARMUP_EPOCHS
    scheduler    = get_scheduler(optimizer, warmup_steps, total_steps)
    scaler       = torch.amp.GradScaler('cuda')

    best_flood_iou = 0.0
    history        = []
    patience_cnt   = 0
    PATIENCE       = 7

    for epoch in range(CFG.EPOCHS):

        # ── Unfreeze backbone after warm-up ──────────────────────────────
        if is_prithvi and epoch == frozen_epochs:
            for p in model.backbone.parameters():
                p.requires_grad = True
            # Re-create optimizer with lower LR for backbone
            optimizer = torch.optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': lr * 0.1},
                {'params': [p for n, p in model.named_parameters()
                            if 'backbone' not in n], 'lr': lr},
            ], weight_decay=CFG.WEIGHT_DECAY)
            scheduler = get_scheduler(optimizer, 0,
                (len(tr_loader) // CFG.ACCUMULATE_STEPS) * (CFG.EPOCHS - frozen_epochs))
            scaler    = torch.amp.GradScaler('cuda')
            print(f'  Backbone unfrozen at epoch {epoch+1} (backbone LR = {lr*0.1:.2e})')

        train_loss  = train_one_epoch(model, tr_loader, optimizer, scheduler, scaler)
        val_metrics = validate(model, va_loader)

        flood_iou = val_metrics['iou_flood']
        history.append({'epoch': epoch+1, 'train_loss': train_loss, **val_metrics})

        print(f'Ep {epoch+1:02d}/{CFG.EPOCHS} | '
              f'TrLoss: {train_loss:.4f} | ValLoss: {val_metrics["val_loss"]:.4f} | '
              f'mIoU: {val_metrics["mIoU"]:.4f} | FloodIoU: {flood_iou:.4f}')

        if flood_iou > best_flood_iou:
            best_flood_iou = flood_iou
            torch.save(model.state_dict(), os.path.join(CFG.SAVE_DIR, save_name))
            print(f'  ✅ Saved (FloodIoU={flood_iou:.4f})')
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  ⏹  Early stop at epoch {epoch+1}')
                break

    print(f'Best Flood IoU: {best_flood_iou:.4f}')
    return pd.DataFrame(history)

print('Training loop defined ✅')

Training loop defined ✅


In [16]:
# ── Simple training (no K-Fold) ──────────────────────────────────────────
train_ds = FloodDataset(train_imgs, train_lbls, transform=train_transform)
val_ds   = FloodDataset(val_imgs,   val_lbls,   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# ── Train Prithvi ─────────────────────────────────────────────────────────
mA = PrithviSegmentation().to(CFG.DEVICE)
hA = train_model(mA, "Prithvi-100M", "best_prithvi.pth",
                 train_loader, val_loader, lr=5e-5, frozen_epochs=3)
del mA; torch.cuda.empty_cache(); gc.collect()

# ── Train UNet++ B5 ───────────────────────────────────────────────────────
mB = build_smp_model_1().to(CFG.DEVICE)
hB = train_model(mB, "UNet++ B5", "best_smp1.pth",
                 train_loader, val_loader, lr=1.5e-4, frozen_epochs=0)
del mB; torch.cuda.empty_cache(); gc.collect()

# ── Train DeepLabV3+ ──────────────────────────────────────────────────────
mC = build_smp_model_2().to(CFG.DEVICE)
hC = train_model(mC, "DeepLabV3+ R50", "best_smp2.pth",
                 train_loader, val_loader, lr=1.5e-4, frozen_epochs=0)
del mC; torch.cuda.empty_cache(); gc.collect()

print(f"\nPrithvi best FloodIoU : {hA['iou_flood'].max():.4f}")
print(f"UNet++ best FloodIoU  : {hB['iou_flood'].max():.4f}")
print(f"DeepLabV3+ best FloodIoU: {hC['iou_flood'].max():.4f}")

Train: 59 | Val: 10
Loading Prithvi-100M weights manually...


Prithvi_100M.pt:   0%|          | 0.00/454M [00:00<?, ?B/s]

Pretrained weights loaded ✅
Prithvi-100M loaded successfully ✅

  Training: Prithvi-100M


Ep 01/25 | TrLoss: 1.0831 | ValLoss: 1.1290 | mIoU: 0.0592 | FloodIoU: 0.0881
  ✅ Saved (FloodIoU=0.0881)


Ep 02/25 | TrLoss: 1.0129 | ValLoss: 1.9835 | mIoU: 0.0315 | FloodIoU: 0.0873


...

Ep 21/25 | TrLoss: 0.6163 | ValLoss: 1.0625 | mIoU: 0.2836 | FloodIoU: 0.1624
  ✅ Saved (FloodIoU=0.1624)


Ep 22/25 | TrLoss: 0.5918 | ValLoss: 1.0513 | mIoU: 0.2694 | FloodIoU: 0.1596


Ep 23/25 | TrLoss: 0.6201 | ValLoss: 0.9871 | mIoU: 0.3055 | FloodIoU: 0.1540


Ep 24/25 | TrLoss: 0.6058 | ValLoss: 1.0189 | mIoU: 0.2901 | FloodIoU: 0.1589


Ep 25/25 | TrLoss: 0.5845 | ValLoss: 1.0887 | mIoU: 0.2434 | FloodIoU: 0.1591
Best Flood IoU: 0.1624

Prithvi best FloodIoU : 0.1195
UNet++ best FloodIoU  : 0.1421
DeepLabV3+ best FloodIoU: 0.1624


In [17]:
print(f"Prithvi  best FloodIoU : {hA['iou_flood'].max():.4f} @ ep{hA['iou_flood'].idxmax()+1}")
print(f"UNet++ B5 best FloodIoU : {hB['iou_flood'].max():.4f} @ ep{hB['iou_flood'].idxmax()+1}")
print(f"DeepLabV3+ best FloodIoU: {hC['iou_flood'].max():.4f} @ ep{hC['iou_flood'].idxmax()+1}")

Prithvi  best FloodIoU : 0.1195 @ ep10
UNet++ B5 best FloodIoU : 0.1421 @ ep15
DeepLabV3+ best FloodIoU: 0.1624 @ ep21


In [18]:
# ── 18. TTA Functions ────────────────────────────────────────────────────────
def apply_tta(image, aug_id):
    img = image.copy()
    if aug_id == 0: return img
    if aug_id == 1: return np.flip(img, axis=2).copy()
    if aug_id == 2: return np.flip(img, axis=1).copy()
    if aug_id == 3: return np.rot90(img, 1, (1, 2)).copy()
    if aug_id == 4: return np.rot90(img, 2, (1, 2)).copy()
    if aug_id == 5: return np.rot90(img, 3, (1, 2)).copy()
    if aug_id == 6:
        return np.rot90(np.flip(img, axis=2), 1, (1, 2)).copy()
    if aug_id == 7:
        return np.rot90(np.flip(img, axis=1), 1, (1, 2)).copy()


def reverse_tta(pred, aug_id):
    if aug_id == 0: return pred
    if aug_id == 1: return np.flip(pred, axis=2).copy()
    if aug_id == 2: return np.flip(pred, axis=1).copy()
    if aug_id == 3: return np.rot90(pred, -1, (1, 2)).copy()
    if aug_id == 4: return np.rot90(pred, -2, (1, 2)).copy()
    if aug_id == 5: return np.rot90(pred, -3, (1, 2)).copy()
    if aug_id == 6:
        return np.flip(np.rot90(pred, -1, (1, 2)), axis=2).copy()
    if aug_id == 7:
        return np.flip(np.rot90(pred, -1, (1, 2)), axis=1).copy()


@torch.no_grad()
def predict_with_tta(model, image_np, tta_ids=None):
    model.eval()
    if tta_ids is None: tta_ids = CFG.TTA_IDS
    all_logits = []
    for aug_id in tta_ids:
        img_aug = apply_tta(image_np, aug_id)
        tensor  = torch.tensor(img_aug, dtype=torch.float32).unsqueeze(0).to(CFG.DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(tensor).cpu().float().numpy()[0]
        all_logits.append(reverse_tta(logits, aug_id))
    return np.mean(all_logits, axis=0)

print('TTA functions defined ✅  (8 transforms, logit-level)')

TTA functions defined ✅  (8 transforms, logit-level)


In [19]:
import cv2

# ── Load best checkpoints ─────────────────────────────────────────────────
iouA = float(hA['iou_flood'].max())
iouB = float(hB['iou_flood'].max())
iouC = float(hC['iou_flood'].max())

mA = PrithviSegmentation().to(CFG.DEVICE)
mA.load_state_dict(torch.load(os.path.join(CFG.SAVE_DIR, "best_prithvi.pth"),
                               map_location=CFG.DEVICE))
mA.eval()

mB = build_smp_model_1().to(CFG.DEVICE)
mB.load_state_dict(torch.load(os.path.join(CFG.SAVE_DIR, "best_smp1.pth"),
                               map_location=CFG.DEVICE))
mB.eval()

mC = build_smp_model_2().to(CFG.DEVICE)
mC.load_state_dict(torch.load(os.path.join(CFG.SAVE_DIR, "best_smp2.pth"),
                               map_location=CFG.DEVICE))
mC.eval()

all_fold_models  = [mA, mB, mC]

# ── IoU-based weights ─────────────────────────────────────────────────────
raw_weights      = [iouA, iouB, iouC]
total_w          = sum(raw_weights)
all_fold_weights = [w / total_w for w in raw_weights]

print(f"Prithvi  weight: {all_fold_weights[0]:.4f}")
print(f"UNet++ B5 weight: {all_fold_weights[1]:.4f}")
print(f"DeepLabV3+ weight: {all_fold_weights[2]:.4f}")

# ── Threshold search on VAL ONLY — no leakage ─────────────────────────────
raw_t           = find_best_threshold(all_fold_models, all_fold_weights,
                                      val_loader, CFG.DEVICE)
FLOOD_THRESHOLD = float(np.clip(raw_t, 0.25, 0.75))
print(f"Threshold: raw={raw_t:.3f}  final={FLOOD_THRESHOLD:.3f}")

Loading Prithvi-100M weights manually...
Pretrained weights loaded ✅
Prithvi-100M loaded successfully ✅
Prithvi  weight: 0.2818
UNet++ B5 weight: 0.3352
DeepLabV3+ weight: 0.3830
Best Threshold: 0.550 | Global Flood IoU: 0.1742
Threshold: raw=0.550  final=0.550


In [20]:
# ── 20. RLE Utilities ────────────────────────────────────────────────────────
def rle_encode(mask):
    mask = mask.astype(np.uint8)
    if mask.sum() == 0: return '0 0'
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs))


def rle_decode(rle_str, shape=(512, 512)):
    if rle_str in ['0 0', '', None]: return np.zeros(shape, dtype=np.uint8)
    nums    = list(map(int, rle_str.split()))
    starts  = np.array(nums[0::2]) - 1
    lengths = np.array(nums[1::2])
    flat    = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for s, l in zip(starts, lengths): flat[s:s+l] = 1
    return flat.reshape(shape, order='F')

print('RLE functions defined ✅')

RLE functions defined ✅


In [21]:
# ── 21. Inference ────────────────────────────────────────────────────────────
class TestDataset(Dataset):
    def __init__(self, img_paths):
        self.img_paths = img_paths
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as f:
            img = f.read().astype(np.float32)
        # Ensure spatial consistency with training
        img = img[:, :CFG.IMG_SIZE, :CFG.IMG_SIZE]
        return torch.tensor(preprocess(img), dtype=torch.float32)

test_ds    = TestDataset(test_imgs)
submission = []
kernel_3   = np.ones((3, 3), np.uint8)
kernel_5   = np.ones((5, 5), np.uint8)

print(f'Inference on {len(test_imgs)} images')
print(f'  Models    : {len(all_fold_models)} architectures')
print(f'  TTA       : {len(CFG.TTA_IDS)} transforms')
print(f'  Threshold : {FLOOD_THRESHOLD:.3f}')

for i, img_path in enumerate(tqdm(test_imgs, desc='Inference')):
    # Extract the preprocessed 6-channel image
    img_np = test_ds[i].numpy()   # (6, H, W)

    # 1. Ensemble logits across all models with TTA
    # all_fold_weights should sum to 1.0 to maintain logit scale
    logits_sum = None
    for model, w in zip(all_fold_models, all_fold_weights):
        logits_tta = predict_with_tta(model, img_np, CFG.TTA_IDS)
        if logits_sum is None:
            logits_sum = w * logits_tta
        else:
            logits_sum += w * logits_tta

    # 2. Softmax to get probabilities for all 3 classes
    # Class 0: No-Flood, Class 1: Flood, Class 2: Water-Body
    probs = torch.softmax(
        torch.tensor(logits_sum, dtype=torch.float32).unsqueeze(0), dim=1
    ).squeeze(0).numpy()
    
    # We focus specifically on Class 1 (Flood) for RLE submission
    flood_prob = probs[1]   # (H, W)

    # 3. Apply the optimized Threshold
    raw_mask = (flood_prob > FLOOD_THRESHOLD).astype(np.uint8)

    # 4. Morphological clean-up to remove small noise and fill gaps
    raw_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN,  kernel_3)
    raw_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_CLOSE, kernel_5)

    # 5. Connected-component filter: remove small islands < 64 pixels
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(raw_mask, connectivity=8)
    clean_mask = np.zeros_like(raw_mask)
    for lbl in range(1, num_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= 64:
            clean_mask[labels == lbl] = 1

    # 6. RLE Encode: mask_to_rle handles the "0 0" for empty masks internally
    # Ensure this uses Class 1 only as per Phase 2 rules
    rle = rle_encode(clean_mask)
    if not rle or rle.strip() == "":
        rle = "0 0"

    submission.append({'id': test_ids[i], 'rle_mask': rle})



Inference on 19 images
  Models    : 3 architectures
  TTA       : 8 transforms
  Threshold : 0.550


Inference: 100%|██████████| 19/19 [00:27<00:00,  1.43s/it]


In [22]:
# ── 22. Save Submission ──────────────────────────────────────────────────────
df = pd.DataFrame(submission, columns=['id', 'rle_mask'])

# Mandatory validations for Phase 2 submission
assert len(df) == len(test_imgs),         '❌ Missing predictions'
assert df['id'].nunique() == len(df),     '❌ Duplicate IDs'
assert df['rle_mask'].isna().sum() == 0,  '❌ Null values found'
assert (df['rle_mask'] == '').sum() == 0, '❌ Empty strings found (use "0 0")'

n_flood   = (df['rle_mask'] != '0 0').sum()
n_noflood = (df['rle_mask'] == '0 0').sum()

print(f'\nTotal Images : {len(df)}')
print(f'Flood Detected : {n_flood} ({100*n_flood/len(df):.1f}%)')
print(f'No-Flood (0 0) : {n_noflood} ({100*n_noflood/len(df):.1f}%)')

out_path = os.path.join(CFG.SAVE_DIR, 'submission.csv')
df.to_csv(out_path, index=False)
print(f'\n🚀 submission.csv saved → {out_path}')


Total Images : 19
Flood Detected : 15 (78.9%)
No-Flood (0 0) : 4 (21.1%)

🚀 submission.csv saved → /kaggle/working/submission.csv


In [23]:
# ── 23. Sanity Check ─────────────────────────────────────────────────────────
for row in df.sample(min(3, len(df))).itertuples():
    if row.rle_mask == '0 0':
        print(f'{row.id}: empty mask (no flood)')
        continue
    decoded = rle_decode(row.rle_mask)
    re_enc  = rle_encode(decoded)
    match   = re_enc == row.rle_mask
    print(f'{row.id}: flood_pixels={decoded.sum()}, round-trip={"✅" if match else "❌"}')

print('\n✅ All checks passed. Ready for submission!')
print(f'Threshold  : {FLOOD_THRESHOLD}')
print(f'Flood count: {n_flood}')

20240529_EO4_RES2_fl_pid_080: empty mask (no flood)
20240529_EO4_RES2_fl_pid_085: flood_pixels=141950, round-trip=✅
20240529_EO4_RES2_fl_pid_091: empty mask (no flood)

✅ All checks passed. Ready for submission!
Threshold  : 0.55
Flood count: 15
